# TinyGPT v2 — SFT Training in `stage4/sft` (Mix A / Mix B)

This notebook fine-tunes two independent copies of the **Stage 4 best checkpoint** using the prepared SFT dataset.

- **Base checkpoint:** `/kaggle/input/notebooks/fakesiddharth/pretrain-stage-4/checkpoints/stage4_best.pt`
- **Prepared data:** `/kaggle/input/notebooks/fakesiddharth/tiny-gpt-data-prep-stage-2/sft_data` (must be attached as a Kaggle input; contains `train.jsonl` and `dev.jsonl`)
- **Mix A:** 70% chat / 30% Q&A by supervised response-token budget.
- **Mix B:** 30% chat / 70% Q&A by supervised response-token budget.
- GPT-2 tokenizer; 256-token context; 8 layers, 8 heads, 512 hidden size (~51.08M parameters).
- Response-only loss: prompt and padding positions are ignored; assistant reply tokens and EOS are trained.
- Adapts the finalized Step 1 JSONL schema by deriving missing `labels`, `active_tokens`, and `split_group_id`.
- Keeps every `handwritten` / `synthetic_recall` example in both mixes before allocating the remaining chat-vs-short-QA token budget.
- If the Stage 4 `validation_cache.pt` is attached, compares base, Mix A, and Mix B FineWeb loss against the ~3.6472 Stage 4 reference.
- Both mixes start independently from the same untouched Stage 4 best weights. Mix B does not start from Mix A.

## Output ownership

All checkpoints, reports and generation results are written under this notebook session's `/kaggle/working` directory:

`/kaggle/working/stage4_sft_run_20261023/`

They are **not** written to the older `/kaggle/working/tinygpt_sft/` directory. After the run completes, use **Save Version** with outputs enabled so this notebook's artifacts can be attached to the testing notebook.

## Before Run All

1. Enable a GPU accelerator.
2. Ensure the `pretrain-stage-4` notebook output containing `stage4_best.pt` is attached.
3. Also attach the completed data-preparation notebook output containing `sft_data/train.jsonl` and `sft_data/dev.jsonl`.
4. Enable Internet if the GPT-2 tokenizer is not already cached.

The first code cell verifies the exact checkpoint path and the prepared data before training. The training configuration and Mix A/Mix B composition are kept the same as the prior SFT notebook; only the input selection/output destination are set for this notebook.


In [1]:
# CELL 1 — Environment, configuration, input discovery, tokenizer and checkpoint.
import os, gc, json, time, math, random, hashlib, traceback, re
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Config, GPT2LMHeadModel, AutoTokenizer, get_cosine_schedule_with_warmup

CFG = {
    "seed": 20261020,
    "context": 256,
    "epochs": 2,
    "per_gpu_batch": 16,
    "grad_accum": 2,
    "lr": 5e-5,
    "min_lr": 5e-6,
    "warmup_fraction": 0.05,
    "weight_decay": 0.01,
    "grad_clip": 1.0,
    "target_active_tokens_epoch": 1_200_000,
    "min_active_tokens_epoch": 100_000,
    "log_every": 25,
    "validate_every": 100,
    "num_workers": 0,
    "output_dir": "/kaggle/working/stage4_sft_run_20261023",
    "ntfy_topic": "tinygpt-siddharth-48271",
    # Set these explicitly only if automatic discovery reports ambiguity.
    "base_checkpoint_path": "/kaggle/input/notebooks/fakesiddharth/pretrain-stage-4/checkpoints/stage4_best.pt",
    "data_dir": "/kaggle/input/notebooks/fakesiddharth/tiny-gpt-data-prep-stage-2/sft_data",
}
SEED = CFG["seed"]
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Input paths are preferences, not hard requirements: discovery below searches /kaggle/input if they differ.
Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)


def notify(title, message):
    try:
        import requests
        r = requests.post("https://ntfy.sh/" + CFG["ntfy_topic"],
                          data=str(message).encode("utf-8"),
                          headers={"Title": str(title).encode("ascii", "replace").decode("ascii")[:100]},
                          timeout=5)
        if not r.ok:
            print(f"[ntfy warning] HTTP {r.status_code}")
        else:
            print("[ntfy]", title, "HTTP", r.status_code)
    except Exception as exc:
        print("[ntfy skipped]", type(exc).__name__, str(exc)[:160])


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("No CUDA GPU detected. In Kaggle, Settings → Accelerator → GPU, then restart and Run All.")
GPU_COUNT = torch.cuda.device_count()
print("PyTorch:", torch.__version__, "| device:", DEVICE, "| GPUs:", GPU_COUNT)
for i in range(GPU_COUNT):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {torch.cuda.get_device_properties(i).total_memory/1024**3:.1f} GiB")


def files_named(root, name):
    root = Path(root)
    return sorted(p for p in root.rglob(name) if p.is_file()) if root.exists() else []


def find_stage4_checkpoint():
    """Use the configured Stage 4 checkpoint when present; otherwise search attached Kaggle inputs."""
    explicit = CFG.get("base_checkpoint_path")
    if explicit:
        p = Path(explicit)
        if p.is_file():
            if p.name != "stage4_best.pt":
                raise RuntimeError("Expected Stage 4 best weights named stage4_best.pt; refusing a Stage 3 fallback.")
            return p
        print(f"[input discovery] Preferred checkpoint path is absent; searching /kaggle/input: {p}")

    matches = files_named("/kaggle/input", "stage4_best.pt")
    # Prefer explicitly identifiable Stage 4 outputs. Never substitute Stage 3 weights.
    preferred = [p for p in matches if "stage4" in str(p).lower() or "pretrain-stage-4" in str(p).lower()]
    candidates = preferred or matches
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Could not uniquely discover stage4_best.pt under /kaggle/input. Found: " +
            ("; ".join(map(str, candidates[:20])) if candidates else "none") +
            ". Attach the Stage 4 output; if multiple Stage 4 copies exist, set CFG['base_checkpoint_path'] to the intended one."
        )
    print("[input discovery] Using discovered Stage 4 checkpoint:", candidates[0])
    return candidates[0]


def find_data_dir():
    """Use the configured SFT data folder when present, else discover train/dev JSONL recursively."""
    explicit = CFG.get("data_dir")
    if explicit:
        p = Path(explicit)
        if (p / "train.jsonl").is_file() and (p / "dev.jsonl").is_file():
            return p
        print(f"[input discovery] Preferred data_dir is absent/incomplete; searching /kaggle/input: {p}")

    dirs = set()
    for root, _, files in os.walk("/kaggle/input"):
        if {"train.jsonl", "dev.jsonl"}.issubset(set(files)):
            dirs.add(Path(root))
    candidates = sorted(dirs, key=lambda p: (not (p / "manifest.json").exists(), "sft_data" not in p.parts, len(str(p))))
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Could not uniquely discover a prepared data directory containing train.jsonl and dev.jsonl. Found: " +
            ("; ".join(map(str, candidates[:20])) if candidates else "none") +
            ". Attach the Step 1 output; if multiple data folders exist, set CFG['data_dir'] explicitly."
        )
    print("[input discovery] Using discovered prepared data directory:", candidates[0])
    return candidates[0]


def find_optional_validation_cache(checkpoint_path):
    """Find the Stage 4 FineWeb validation cache if attached; it is optional for SFT."""
    exact = Path(checkpoint_path).parent / "validation_cache.pt"
    if exact.is_file():
        return exact
    matches = files_named("/kaggle/input", "validation_cache.pt")
    if not matches:
        return None
    stage4_matches = [p for p in matches if "pretrain-stage-4" in str(p).lower() or "stage4" in str(p).lower()]
    candidates = stage4_matches or matches
    if len(candidates) == 1:
        return candidates[0]
    print("[drift check skipped] Multiple validation_cache.pt files found; refusing to guess:")
    for p in candidates[:20]: print("  ", p)
    return None

BASE_PATH = find_stage4_checkpoint()
DATA_DIR = find_data_dir()
FINEWEB_VALIDATION_CACHE_PATH = find_optional_validation_cache(BASE_PATH)
OUT_DIR = Path(CFG["output_dir"]); OUT_DIR.mkdir(parents=True, exist_ok=True)
if FINEWEB_VALIDATION_CACHE_PATH is None:
    print("Optional Stage 4 validation_cache.pt not found/unambiguous; FineWeb drift check will be skipped.")
else:
    print("Optional FineWeb drift-check cache:", FINEWEB_VALIDATION_CACHE_PATH)
print("CURRENT RUN OUTPUT DIRECTORY:", OUT_DIR.resolve())
print("All Mix A/Mix B checkpoints and reports will be written here; no output is written to the older tinygpt_sft directory.")
print("Stage 4 checkpoint:", BASE_PATH, f"({BASE_PATH.stat().st_size/1024**2:.1f} MiB)")
print("Prepared data:", DATA_DIR)

try:
    tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
except Exception as exc:
    raise RuntimeError("Could not load GPT-2 tokenizer. Enable Internet or attach/cache the GPT-2 tokenizer. " + repr(exc)) from exc
if len(tokenizer) != 50257 or tokenizer.eos_token_id != 50256:
    raise RuntimeError(f"Unexpected GPT-2 tokenizer: vocab={len(tokenizer)}, eos={tokenizer.eos_token_id}")
EOS = tokenizer.eos_token_id

MODEL_CONFIG = GPT2Config(
    vocab_size=50257, n_positions=256, n_ctx=256, n_embd=512,
    n_layer=8, n_head=8, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
    bos_token_id=EOS, eos_token_id=EOS, pad_token_id=EOS, use_cache=False,
)

def unwrap_state(obj):
    if not isinstance(obj, dict): raise TypeError(f"Checkpoint must be a dict, got {type(obj)}")
    for key in ("model_state_dict", "state_dict", "model", "model_weights", "weights"):
        value = obj.get(key)
        if isinstance(value, dict) and value and all(torch.is_tensor(x) for x in value.values()):
            obj = value; break
    if not obj or not all(torch.is_tensor(v) for v in obj.values()):
        raise RuntimeError("No recognizable tensor state_dict found in stage4_best.pt")
    if all(k.startswith("module.") for k in obj): obj = {k[7:]: v for k, v in obj.items()}
    return obj

try:
    try: payload = torch.load(BASE_PATH, map_location="cpu", weights_only=False)
    except TypeError: payload = torch.load(BASE_PATH, map_location="cpu")
    BASE_STATE = unwrap_state(payload)
except Exception as exc:
    raise RuntimeError(f"Failed reading Stage 4 checkpoint {BASE_PATH}: {exc}") from exc

BASE_MODEL = GPT2LMHeadModel(MODEL_CONFIG)
missing, unexpected = BASE_MODEL.load_state_dict(BASE_STATE, strict=False)
if missing or unexpected:
    raise RuntimeError(f"Stage 4/model architecture mismatch. Missing={missing[:8]}, unexpected={unexpected[:8]}")
PARAM_COUNT = sum(p.numel() for p in BASE_MODEL.parameters())
if PARAM_COUNT != 51_082_752:
    raise RuntimeError(f"Wrong TinyGPT v2 architecture: expected 51,082,752 params, got {PARAM_COUNT:,}")
print(f"Checkpoint verified: {PARAM_COUNT:,} parameters; state dict matches architecture exactly.")
BASE_STATE = {k: v.detach().cpu().contiguous().clone() for k, v in BASE_MODEL.state_dict().items()}
del BASE_MODEL, payload, missing, unexpected
gc.collect(); torch.cuda.empty_cache()
notify("TinyGPT SFT training ready", f"Stage4 checkpoint verified; prepared data={DATA_DIR}")


PyTorch: 2.11.0+cu128 | device: cuda | GPUs: 2
GPU 0: Tesla T4 | 14.6 GiB
GPU 1: Tesla T4 | 14.6 GiB
Optional FineWeb drift-check cache: /kaggle/input/notebooks/fakesiddharth/pretrain-stage-4/checkpoints/validation_cache.pt
CURRENT RUN OUTPUT DIRECTORY: /kaggle/working/stage4_sft_run_20261023
All Mix A/Mix B checkpoints and reports will be written here; no output is written to the older tinygpt_sft directory.
Stage 4 checkpoint: /kaggle/input/notebooks/fakesiddharth/pretrain-stage-4/checkpoints/stage4_best.pt (293.1 MiB)
Prepared data: /kaggle/input/notebooks/fakesiddharth/tiny-gpt-data-prep-stage-2/sft_data


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Checkpoint verified: 51,082,752 parameters; state dict matches architecture exactly.
[ntfy] TinyGPT SFT training ready HTTP 200


In [2]:
# CELL 2 — Read and validate the prepared data; build fair A/B mixes.

def derive_step1_fields(row, path, line_no):
    """Adapt Step 1 export records to the trainer schema without requiring extra JSONL fields."""
    if not isinstance(row, dict):
        raise RuntimeError(f"Expected a JSON object in {path}, line {line_no}; got {type(row).__name__}")

    if "labels" not in row:
        if "prompt_length" not in row or "input_ids" not in row:
            raise RuntimeError(f"Cannot derive labels in {path}, line {line_no}: input_ids/prompt_length missing")
        prompt_len = int(row["prompt_length"])
        ids = row["input_ids"]
        if not isinstance(ids, list) or prompt_len < 0 or prompt_len > len(ids):
            raise RuntimeError(f"Invalid input_ids/prompt_length in {path}, line {line_no}")
        row["labels"] = [-100] * prompt_len + list(ids[prompt_len:])

    if "active_tokens" not in row:
        if "target_length" not in row:
            raise RuntimeError(f"Cannot derive active_tokens in {path}, line {line_no}: target_length missing")
        row["active_tokens"] = int(row["target_length"])

    if not row.get("split_group_id"):
        conversation_id = re.sub(r":rep\d+$", "", str(row.get("conversation_id", "")))
        if str(row.get("source", "")) == "oasst1":
            match = re.match(r"^oasst:([^:]+):[^:]+$", conversation_id)
            if match:
                row["split_group_id"] = f"oasst-tree:{match.group(1)}"
            else:
                row["split_group_id"] = conversation_id
        else:
            row["split_group_id"] = conversation_id
    return row


def load_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip(): continue
            try: row = json.loads(line)
            except json.JSONDecodeError as exc: raise RuntimeError(f"Invalid JSON in {path}, line {line_no}: {exc}") from exc
            rows.append(derive_step1_fields(row, path, line_no))
    return rows

train_all = load_jsonl(DATA_DIR / "train.jsonl")
dev_all = load_jsonl(DATA_DIR / "dev.jsonl")
if not train_all or not dev_all:
    raise RuntimeError(f"Prepared data is empty: train={len(train_all):,}, dev={len(dev_all):,}")

def validate_record(ex, name, idx):
    required = {"input_ids", "labels", "prompt_length", "target_length", "active_tokens", "category", "source", "conversation_id", "split_group_id"}
    missing = required - set(ex)
    if missing: raise RuntimeError(f"{name}[{idx}] missing keys: {sorted(missing)}")
    ids, labels = ex["input_ids"], ex["labels"]
    if not isinstance(ids, list) or not isinstance(labels, list) or len(ids) != len(labels) or not ids:
        raise RuntimeError(f"{name}[{idx}] invalid input_ids/labels")
    if len(ids) > CFG["context"]: raise RuntimeError(f"{name}[{idx}] exceeds context length")
    if any(not isinstance(t, int) or t < 0 or t >= 50257 for t in ids): raise RuntimeError(f"{name}[{idx}] has invalid token IDs")
    if labels[-1] != EOS: raise RuntimeError(f"{name}[{idx}] EOS is not supervised")
    active = sum(v != -100 for v in labels)
    if active < 2 or active != int(ex["active_tokens"]): raise RuntimeError(f"{name}[{idx}] bad active-token count; expect at least one reply token plus EOS")
    first = next((i for i,v in enumerate(labels) if v != -100), None)
    if first is None or any(v != -100 for v in labels[:first]) or any(v == -100 for v in labels[first:]):
        raise RuntimeError(f"{name}[{idx}] response-only label mask is invalid")
    if len(ids) - first != active: raise RuntimeError(f"{name}[{idx}] prompt/response boundary mismatch")

for name, rows in (("train", train_all), ("dev", dev_all)):
    for i, ex in enumerate(rows): validate_record(ex, name, i)

# Split membership must be consistent for each original conversation/tree across the files.
train_groups = {str(x["split_group_id"]) for x in train_all}
dev_groups = {str(x["split_group_id"]) for x in dev_all}
overlap = train_groups & dev_groups
if overlap:
    raise RuntimeError(f"Leakage detected: {len(overlap)} split_group_id(s) appear in train and dev; e.g. {next(iter(overlap))}")

CHAT_CATEGORIES = {"casual_chat", "emotional_chat"}
QA_CATEGORIES = {"short_qa"}

def is_preserved_identity_example(ex):
    # Keep all intentional personal/identity and synthetic recall examples in BOTH mixes.
    return str(ex.get("category", "")) == "handwritten" or str(ex.get("source", "")) == "synthetic_recall"

def kind(ex):
    cat = str(ex.get("category", ""))
    if is_preserved_identity_example(ex): return "handwritten"
    if cat in CHAT_CATEGORIES: return "chat"
    if cat in QA_CATEGORIES: return "qa"
    raise RuntimeError(f"Unknown prepared category/source combination: category={cat!r}, source={ex.get('source')!r}; update mix mapping explicitly.")

chat_train = [x for x in train_all if kind(x) == "chat"]
qa_train = [x for x in train_all if kind(x) == "qa"]
handwritten_train = [x for x in train_all if kind(x) == "handwritten"]
chat_dev = [x for x in dev_all if kind(x) == "chat"]
qa_dev = [x for x in dev_all if kind(x) == "qa"]
handwritten_dev = [x for x in dev_all if kind(x) == "handwritten"]
if not chat_train or not qa_train or not chat_dev or not qa_dev:
    raise RuntimeError(f"Both chat and short-QA need non-empty train/dev pools. train chat={len(chat_train)}, short_qa={len(qa_train)}; dev chat={len(chat_dev)}, short_qa={len(qa_dev)}")

print("Prepared records:", len(train_all), "train /", len(dev_all), "dev")
print("Train categories:", dict(Counter(x['category'] for x in train_all)))
print("Train sources:", dict(Counter(x['source'] for x in train_all)))
print("Pool counts: chat train/dev =", len(chat_train), "/", len(chat_dev), "| QA train/dev =", len(qa_train), "/", len(qa_dev))
print("Supervised tokens: train=", sum(x['active_tokens'] for x in train_all), "dev=", sum(x['active_tokens'] for x in dev_all))
if (DATA_DIR / "manifest.json").exists():
    try:
        manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
        print("Data prep created:", manifest.get("created_utc", "unknown"))
    except Exception as exc: print("Manifest read warning:", repr(exc))


def choose_by_active_tokens(pool, budget, seed):
    items = list(pool)
    random.Random(seed).shuffle(items)
    chosen = []; total = 0
    for item in items:
        if total >= budget: break
        chosen.append(item); total += int(item["active_tokens"])
    return chosen, total


def build_mix(chat_ratio, seed):
    """Build the requested chat/short-QA ratio while retaining every identity/recall row."""
    qa_ratio = 1.0 - chat_ratio
    chat_cap = sum(int(x["active_tokens"]) for x in chat_train)
    qa_cap = sum(int(x["active_tokens"]) for x in qa_train)
    handwritten_tokens = sum(int(x["active_tokens"]) for x in handwritten_train)

    # Handwritten/synthetic-recall examples are included 100% in both mixes first.
    remaining_cap = int(CFG["target_active_tokens_epoch"]) - handwritten_tokens
    if remaining_cap <= 0:
        raise RuntimeError(
            f"Preserved handwritten/recall pool uses {handwritten_tokens:,} tokens, which leaves no room "
            f"under target_active_tokens_epoch={CFG['target_active_tokens_epoch']:,}."
        )
    feasible_remaining = min(remaining_cap, int(chat_cap / chat_ratio), int(qa_cap / qa_ratio))
    feasible_total = handwritten_tokens + feasible_remaining
    if feasible_total < CFG["min_active_tokens_epoch"] or feasible_remaining <= 0:
        raise RuntimeError(
            f"Insufficient clean tokens for {chat_ratio:.0%}/{qa_ratio:.0%} chat/short-QA mix: "
            f"chat={chat_cap:,}, short_QA={qa_cap:,}, preserved handwritten/recall={handwritten_tokens:,}, "
            f"feasible total={feasible_total:,}; need at least {CFG['min_active_tokens_epoch']:,}."
        )

    chat_budget = int(feasible_remaining * chat_ratio)
    qa_budget = feasible_remaining - chat_budget
    c, ct = choose_by_active_tokens(chat_train, chat_budget, seed)
    q, qt = choose_by_active_tokens(qa_train, qa_budget, seed + 1)
    preserved = list(handwritten_train)
    rows = c + q + preserved
    random.Random(seed + 2).shuffle(rows)
    actual_chat_qa = ct + qt
    actual_total = actual_chat_qa + handwritten_tokens
    return {
        "records": rows, "chat_records": c, "qa_records": q,
        "handwritten_records": preserved, "chat_tokens": ct, "qa_tokens": qt,
        "handwritten_tokens": handwritten_tokens, "target_tokens": feasible_total,
        "actual_tokens": actual_total,
        # This ratio intentionally excludes the always-included preserved pool.
        "chat_ratio_actual": ct / max(actual_chat_qa, 1),
        "chat_ratio_overall": ct / max(actual_total, 1),
    }

mixA = build_mix(0.70, SEED + 101)
mixB = build_mix(0.30, SEED + 202)
print(f"Preserved identity/handwritten/synthetic-recall pool: {len(handwritten_train):,} records / {sum(int(x['active_tokens']) for x in handwritten_train):,} active tokens; included in BOTH mixes.")
for name, mix in (("Mix A", mixA), ("Mix B", mixB)):
    ratio = mix["chat_ratio_actual"]
    print(f"{name}: {len(mix['records']):,} examples; chat={len(mix['chat_records']):,}/{mix['chat_tokens']:,} active tokens ({ratio:.1%} of chat+QA); short_QA={len(mix['qa_records']):,}/{mix['qa_tokens']:,}; preserved={len(mix['handwritten_records']):,}/{mix['handwritten_tokens']:,} tokens; total active={mix['actual_tokens']:,}")
    # Every preserved row must appear, including each source=synthetic_recall row even if its category is short_qa.
    assert {id(x) for x in mix["handwritten_records"]} == {id(x) for x in handwritten_train}

# Defensive checks: no train conversation/tree group appears in validation.
for pool in (mixA["records"], mixB["records"]):
    assert not ({str(x["split_group_id"]) for x in pool} & dev_groups)



Prepared records: 83666 train / 10046 dev
Train categories: {'casual_chat': 54000, 'emotional_chat': 18000, 'short_qa': 10716, 'handwritten': 950}
Train sources: {'soda': 39668, 'empathetic_dialogues': 18000, 'dolly': 8766, 'oasst1': 1950, 'daily_dialog': 10131, 'everyday': 4201, 'synthetic_recall': 950}
Pool counts: chat train/dev = 72000 / 9436 | QA train/dev = 10716 / 560
Supervised tokens: train= 1807596 dev= 205609
Data prep created: 2026-10-20T13:23:53.586409+00:00
Preserved identity/handwritten/synthetic-recall pool: 950 records / 13,913 active tokens; included in BOTH mixes.
Mix A: 53,540 examples; chat=43,100/830,270 active tokens (70.0% of chat+QA); short_QA=9,490/355,863; preserved=950/13,913 tokens; total active=1,200,046
Mix B: 20,632 examples; chat=8,966/172,774 active tokens (30.0% of chat+QA); short_QA=10,716/403,051; preserved=950/13,913 tokens; total active=589,738


In [3]:
# CELL 3 — Batching, response-token loss, validation, atomic checkpoints and trainer.
from datetime import datetime, timezone

class SFTDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        r = self.rows[idx]
        return {"input_ids": r["input_ids"], "labels": r["labels"]}

def collate_sft(batch):
    length = max(len(x["input_ids"]) for x in batch)
    ids = torch.full((len(batch), length), EOS, dtype=torch.long)
    labels = torch.full((len(batch), length), -100, dtype=torch.long)
    mask = torch.zeros((len(batch), length), dtype=torch.long)
    for i, row in enumerate(batch):
        n = len(row["input_ids"])
        ids[i, :n] = torch.tensor(row["input_ids"], dtype=torch.long)
        labels[i, :n] = torch.tensor(row["labels"], dtype=torch.long)
        mask[i, :n] = 1
    return {"input_ids": ids, "labels": labels, "attention_mask": mask}

class LossSumModel(nn.Module):
    """Return summed shifted token loss and supervised-token count for DataParallel-safe aggregation."""
    def __init__(self, model): super().__init__(); self.model = model
    def forward(self, input_ids, attention_mask, labels):
        logits = self.model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, return_dict=True).logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss_sum = F.cross_entropy(logits.reshape(-1, logits.size(-1)), shift_labels.reshape(-1), ignore_index=-100, reduction="sum")
        count = (shift_labels != -100).sum().to(loss_sum.dtype)
        return torch.stack((loss_sum, count))


def atomic_torch_save(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def gpu_status():
    if not torch.cuda.is_available(): return "CPU"
    return " | ".join(f"GPU{i} {torch.cuda.memory_allocated(i)/1024**3:.2f}/{torch.cuda.memory_reserved(i)/1024**3:.2f} GiB" for i in range(torch.cuda.device_count()) )

@torch.inference_mode()
def eval_loss(model, rows, batch_size):
    if not rows: return float("nan")
    model.eval(); loader = DataLoader(SFTDataset(rows), batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate_sft)
    total_loss = 0.0; total_tokens = 0
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE); labels = batch["labels"].to(DEVICE); mask = batch["attention_mask"].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(input_ids=ids, attention_mask=mask, use_cache=False, return_dict=True).logits[:, :-1, :].contiguous()
            shifted = labels[:, 1:].contiguous()
            loss_sum = F.cross_entropy(logits.reshape(-1, logits.size(-1)), shifted.reshape(-1), ignore_index=-100, reduction="sum")
        total_loss += float(loss_sum.float().item())
        total_tokens += int((shifted != -100).sum().item())
    return total_loss / max(total_tokens, 1)


def run_mix(name, mix, base_state):
    seed = SEED + (101 if name == "mixA" else 202)
    seed_all(seed)
    out = OUT_DIR / name; out.mkdir(parents=True, exist_ok=True)
    best_path = out / f"sft_{name}_best.pt"
    latest_path = out / f"sft_{name}_latest.pt"
    final_path = out / f"sft_{name}_final.pt"

    model = GPT2LMHeadModel(MODEL_CONFIG)
    model.load_state_dict(base_state, strict=True)
    model.to(DEVICE)
    try:
        model.set_attn_implementation("sdpa")
        print(name, "using PyTorch SDPA attention")
    except Exception: pass
    wrapped = LossSumModel(model)
    if torch.cuda.device_count() > 1: wrapped = nn.DataParallel(wrapped)

    global_batch = CFG["per_gpu_batch"] * max(torch.cuda.device_count(), 1)
    dl = DataLoader(SFTDataset(mix["records"]), batch_size=global_batch, shuffle=True,
                    collate_fn=collate_sft, num_workers=CFG["num_workers"],
                    generator=torch.Generator().manual_seed(seed), pin_memory=True)
    steps_per_epoch = math.ceil(len(dl) / CFG["grad_accum"])
    total_steps = steps_per_epoch * CFG["epochs"]
    warmup_steps = max(1, int(total_steps * CFG["warmup_fraction"]))
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"], betas=(0.9, 0.95))
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler = torch.amp.GradScaler("cuda")
    best_score = float("inf"); global_step = 0; examples_seen = 0; start = time.time(); roll_start = start; roll_tokens = 0
    print(f"\n{name}: epochs={CFG['epochs']} | rows={len(mix['records']):,} | global_batch={global_batch} | steps={total_steps:,} | warmup={warmup_steps}")
    notify(name + " started", f"examples={len(mix['records']):,}; steps={total_steps:,}; chat token share={mix['chat_ratio_actual']:.1%}")

    optimizer.zero_grad(set_to_none=True)
    try:
        for epoch in range(CFG["epochs"]):
            model.train(); pending = []
            for batch_idx, batch in enumerate(dl):
                pending.append(batch)
                if len(pending) < CFG["grad_accum"] and batch_idx < len(dl) - 1: continue
                for mb in pending:
                    ids = mb["input_ids"].to(DEVICE, non_blocking=True)
                    labels = mb["labels"].to(DEVICE, non_blocking=True)
                    mask = mb["attention_mask"].to(DEVICE, non_blocking=True)
                    active_count = (labels[:, 1:] != -100).sum().clamp_min(1)
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        got = wrapped(input_ids=ids, attention_mask=mask, labels=labels)
                        if got.ndim == 2:
                            loss_sum, _ = got[:, 0].sum(), got[:, 1].sum()
                        else:
                            loss_sum, _ = got[0], got[1]
                        loss = loss_sum / active_count / len(pending)
                    scaler.scale(loss).backward()
                    nactive = int(active_count.item()); roll_tokens += nactive; examples_seen += ids.shape[0]
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
                for group in optimizer.param_groups: group["lr"] = max(group["lr"], CFG["min_lr"])
                global_step += 1; pending = []

                if global_step % CFG["log_every"] == 0 or global_step == 1:
                    now = time.time(); rate = roll_tokens / max(now - roll_start, 1e-6)
                    done = epoch * steps_per_epoch + global_step - epoch * steps_per_epoch
                    remaining = max(total_steps - global_step, 0); eta = remaining * (now - start) / max(global_step, 1)
                    print(f"[{name}] epoch={epoch+1}/{CFG['epochs']} step={global_step}/{total_steps} examples={examples_seen:,} active_tok/s={rate:,.0f} lr={optimizer.param_groups[0]['lr']:.2e} elapsed={(now-start)/60:.1f}m ETA={eta/60:.1f}m | {gpu_status()}")
                    roll_start = now; roll_tokens = 0
                    if global_step % (CFG["log_every"] * 4) == 0:
                        notify(name + " progress", f"step={global_step}/{total_steps}; active tokens/s={rate:.0f}; ETA={eta/60:.1f} minutes")

                if global_step % CFG["validate_every"] == 0 or global_step == total_steps:
                    val_chat_loss = eval_loss(model, chat_dev, max(1, CFG["per_gpu_batch"] * max(torch.cuda.device_count(), 1)))
                    val_qa_loss = eval_loss(model, qa_dev, max(1, CFG["per_gpu_batch"] * max(torch.cuda.device_count(), 1)))
                    val_score = val_chat_loss + val_qa_loss
                    metrics = {"epoch": epoch + 1, "step": global_step, "val_chat_loss": val_chat_loss, "val_qa_loss": val_qa_loss, "val_score_sum": val_score}
                    print(f"[{name}] validation: chat={val_chat_loss:.4f} | QA={val_qa_loss:.4f} | sum={val_score:.4f}")
                    if math.isfinite(val_score) and val_score < best_score:
                        best_score = val_score
                        atomic_torch_save(model.state_dict(), best_path)
                        print("Saved best:", best_path)
                    atomic_torch_save({"format_version": 1, "run_name": name, "model_state_dict": model.state_dict(),
                                       "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),
                                       "scaler_state_dict": scaler.state_dict(), "epoch": epoch, "step": global_step,
                                       "metrics": metrics, "config": CFG, "mix": {"chat_ratio_target": 0.70 if name == "mixA" else 0.30,
                                       "chat_ratio_actual_tokens_excluding_preserved": mix["chat_ratio_actual"],
                                       "chat_ratio_overall_tokens": mix["chat_ratio_overall"],
                                       "preserved_identity_examples": len(mix["handwritten_records"]),
                                       "preserved_identity_tokens": mix["handwritten_tokens"],
                                       "examples": len(mix["records"]) }}, latest_path)
                    notify(name + " validation", f"step={global_step}; chat={val_chat_loss:.4f}; QA={val_qa_loss:.4f}")
                    model.train()

        atomic_torch_save(model.state_dict(), final_path)
        final_metrics = {"val_chat_loss": eval_loss(model, chat_dev, global_batch), "val_qa_loss": eval_loss(model, qa_dev, global_batch)}
        print(name, "final:", final_metrics)
        notify(name + " complete", json.dumps(final_metrics))
        return {"best": str(best_path), "latest": str(latest_path), "final": str(final_path), "metrics": final_metrics,
                "best_validation_score": best_score, "examples": len(mix["records"]), "chat_token_share": mix["chat_ratio_actual"]}
    except Exception as exc:
        notify(name + " FAILED", f"step={global_step}; {type(exc).__name__}: {exc}\n{traceback.format_exc()[-2000:]}")
        raise
    finally:
        del wrapped, model, optimizer, scheduler, scaler, dl
        gc.collect(); torch.cuda.empty_cache()


In [4]:
# CELL 4 — Train Mix A from fresh Stage 4 weights.
mixA_result = run_mix("mixA", mixA, BASE_STATE)
print("Mix A result:\n", json.dumps(mixA_result, indent=2))


mixA using PyTorch SDPA attention

mixA: epochs=2 | rows=53,540 | global_batch=32 | steps=1,674 | warmup=83
[ntfy] mixA started HTTP 200
[mixA] epoch=1/2 step=1/1674 examples=64 active_tok/s=458 lr=5.00e-06 elapsed=0.0m ETA=83.3m | GPU0 0.60/3.66 GiB | GPU1 0.02/3.64 GiB
[mixA] epoch=1/2 step=25/1674 examples=1,600 active_tok/s=3,484 lr=1.51e-05 elapsed=0.2m ETA=14.2m | GPU0 0.60/11.57 GiB | GPU1 0.02/10.99 GiB
[mixA] epoch=1/2 step=50/1674 examples=3,200 active_tok/s=3,347 lr=3.01e-05 elapsed=0.4m ETA=12.6m | GPU0 0.60/13.10 GiB | GPU1 0.02/12.52 GiB
[mixA] epoch=1/2 step=75/1674 examples=4,800 active_tok/s=3,420 lr=4.52e-05 elapsed=0.6m ETA=12.0m | GPU0 0.60/13.10 GiB | GPU1 0.02/12.52 GiB
[mixA] epoch=1/2 step=100/1674 examples=6,400 active_tok/s=3,431 lr=5.00e-05 elapsed=0.7m ETA=11.6m | GPU0 0.60/13.10 GiB | GPU1 0.02/12.52 GiB
[ntfy] mixA progress HTTP 200
[mixA] validation: chat=2.8765 | QA=3.2403 | sum=6.1168
Saved best: /kaggle/working/stage4_sft_run_20261023/mixA/sft_mixA_bes

In [5]:
# CELL 5 — Train Mix B independently from the same untouched Stage 4 weights.
gc.collect(); torch.cuda.empty_cache()
mixB_result = run_mix("mixB", mixB, BASE_STATE)
print("Mix B result:\n", json.dumps(mixB_result, indent=2))
notify("TinyGPT dual SFT training done", f"MixA={mixA_result['metrics']}; MixB={mixB_result['metrics']}")


mixB using PyTorch SDPA attention

mixB: epochs=2 | rows=20,632 | global_batch=32 | steps=646 | warmup=32
[ntfy] mixB started HTTP 200
[mixB] epoch=1/2 step=1/646 examples=64 active_tok/s=2,332 lr=5.00e-06 elapsed=0.0m ETA=8.6m | GPU0 0.61/4.00 GiB | GPU1 0.02/3.97 GiB
[mixB] epoch=1/2 step=25/646 examples=1,600 active_tok/s=4,161 lr=3.91e-05 elapsed=0.2m ETA=4.7m | GPU0 0.61/6.28 GiB | GPU1 0.02/5.51 GiB
[mixB] epoch=1/2 step=50/646 examples=3,200 active_tok/s=4,206 lr=4.99e-05 elapsed=0.4m ETA=4.4m | GPU0 0.61/6.28 GiB | GPU1 0.02/5.51 GiB
[mixB] epoch=1/2 step=75/646 examples=4,800 active_tok/s=4,199 lr=4.94e-05 elapsed=0.6m ETA=4.2m | GPU0 0.61/6.28 GiB | GPU1 0.02/5.51 GiB
[mixB] epoch=1/2 step=100/646 examples=6,400 active_tok/s=4,134 lr=4.85e-05 elapsed=0.7m ETA=4.0m | GPU0 0.61/6.28 GiB | GPU1 0.02/5.51 GiB
[ntfy] mixB progress HTTP 200
[mixB] validation: chat=2.9245 | QA=3.1791 | sum=6.1036
Saved best: /kaggle/working/stage4_sft_run_20261023/mixB/sft_mixB_best.pt
[ntfy] mixB v

In [6]:
# CELL 6 — Evaluation: base vs each SFT checkpoint; concise generation report.
PROMPTS = [
    ("chat", "Hi! How has your day been?"),
    ("chat", "I had a difficult day today."),
    ("chat", "What do you like to do in your free time?"),
    ("chat", "Can you help me think through a problem?"),
    ("qa", "What is supervised fine-tuning?"),
    ("qa", "Explain what a GPU does in simple terms."),
    ("qa", "What is the capital of France?"),
    ("qa", "Write a Python function that returns the square of a number."),
    ("qa", "What is photosynthesis?"),
]

def generate(model, prompt, seed=11, sample=False):
    full_prompt = f"User: {prompt}\nAssistant:"
    enc = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=220).to(DEVICE)
    kwargs = {"max_new_tokens": 80, "do_sample": sample, "eos_token_id": EOS, "pad_token_id": EOS,
              "repetition_penalty": 1.12, "use_cache": True}
    if sample:
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        kwargs.update({"temperature": 0.7, "top_p": 0.9})
    model.eval()
    with torch.inference_mode():
        out = model.generate(**enc, **kwargs)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def load_weights_model(path):
    try: st = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError: st = torch.load(path, map_location="cpu")
    st = unwrap_state(st)
    model = GPT2LMHeadModel(MODEL_CONFIG)
    model.load_state_dict(st, strict=True)
    return model.to(DEVICE).eval()

@torch.inference_mode()
def eval_fineweb_cache_loss(model, validation_ids, batch_size=4):
    """Token-weighted causal LM loss on cached 256-token FineWeb validation chunks."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for start in range(0, validation_ids.shape[0], batch_size):
        x = validation_ids[start:start + batch_size].long().to(DEVICE)
        attention_mask = torch.ones_like(x)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            logits = model(input_ids=x, attention_mask=attention_mask, use_cache=False, return_dict=True).logits[:, :-1, :].contiguous()
            targets = x[:, 1:].contiguous()
            loss_sum = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction="sum"
            )
        total_loss += float(loss_sum.float().item())
        total_tokens += int(targets.numel())
    if total_tokens == 0:
        raise RuntimeError("The attached FineWeb validation cache contains no scored tokens.")
    return total_loss / total_tokens

def load_fineweb_cache(path):
    try: obj = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError: obj = torch.load(path, map_location="cpu")
    metadata = None
    if torch.is_tensor(obj):
        ids = obj
    elif isinstance(obj, dict) and torch.is_tensor(obj.get("input_ids")):
        ids = obj["input_ids"]
        metadata = obj.get("metadata")
    else:
        raise RuntimeError("Unsupported validation_cache.pt format; expected a token tensor or a dict with tensor key 'input_ids'.")
    if ids.ndim != 2 or ids.shape[1] != CFG["context"]:
        raise RuntimeError(f"Incompatible validation cache shape {tuple(ids.shape)}; expected [N, {CFG['context']}].")
    if ids.numel() == 0 or int(ids.min()) < 0 or int(ids.max()) >= MODEL_CONFIG.vocab_size:
        raise RuntimeError("Validation cache contains no tokens or token IDs outside the TinyGPT vocabulary.")
    return ids.long().contiguous(), metadata

models = {
    "BASE_STAGE4": load_weights_model(BASE_PATH),
    "MIX_A": load_weights_model(mixA_result["best"]),
    "MIX_B": load_weights_model(mixB_result["best"]),
}

fineweb_drift_result = {
    "available": False,
    "reference_stage4_baseline_loss": 3.6472,
    "cache_path": str(FINEWEB_VALIDATION_CACHE_PATH) if FINEWEB_VALIDATION_CACHE_PATH else None,
}
if FINEWEB_VALIDATION_CACHE_PATH is not None:
    try:
        fineweb_ids, fineweb_metadata = load_fineweb_cache(FINEWEB_VALIDATION_CACHE_PATH)
        print(f"\nFineWeb drift check: {fineweb_ids.shape[0]:,} chunks from {FINEWEB_VALIDATION_CACHE_PATH}")
        scores = {}
        for model_name, model in models.items():
            scores[model_name] = eval_fineweb_cache_loss(model, fineweb_ids)
            print(f"  {model_name} FineWeb validation loss: {scores[model_name]:.4f}")
        fineweb_drift_result.update({
            "available": True,
            "metadata": fineweb_metadata,
            "num_chunks": int(fineweb_ids.shape[0]),
            "num_token_positions": int(fineweb_ids.numel()),
            "losses": scores,
            "base_minus_reference": scores["BASE_STAGE4"] - 3.6472,
            "mixA_minus_base": scores["MIX_A"] - scores["BASE_STAGE4"],
            "mixB_minus_base": scores["MIX_B"] - scores["BASE_STAGE4"],
        })
        print(f"  Reference Stage 4 baseline: ~3.6472; measured base delta: {fineweb_drift_result['base_minus_reference']:+.4f}")
        del fineweb_ids
    except Exception as exc:
        print(f"[drift check warning] Could not evaluate attached validation cache: {type(exc).__name__}: {exc}")
        fineweb_drift_result["error"] = f"{type(exc).__name__}: {exc}"
else:
    print("FineWeb drift check skipped: no unambiguous validation_cache.pt was found under /kaggle/input.")

report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "base_checkpoint": str(BASE_PATH), "data_dir": str(DATA_DIR),
    "config": CFG,
    "mixA": mixA_result, "mixB": mixB_result,
    "fineweb_validation_drift_check": fineweb_drift_result,
    "validation": {
        "base_chat_loss": eval_loss(models["BASE_STAGE4"], chat_dev, 8),
        "base_qa_loss": eval_loss(models["BASE_STAGE4"], qa_dev, 8),
        "mixA": mixA_result["metrics"], "mixB": mixB_result["metrics"],
    },
    "generations": {},
}
for name, model in models.items():
    rows = []
    for cat, prompt in PROMPTS:
        rows.append({"category": cat, "prompt": prompt,
                     "greedy": generate(model, prompt, sample=False),
                     "sampled": generate(model, prompt, seed=SEED + len(rows), sample=True)})
    report["generations"][name] = rows

json_path = OUT_DIR / "sft_comparison_report.json"
txt_path = OUT_DIR / "sft_comparison_report.txt"
json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
with txt_path.open("w", encoding="utf-8") as f:
    f.write("TinyGPT v2 SFT: Stage 4 base vs Mix A vs Mix B\n")
    f.write("\nVALIDATION METRICS\n" + json.dumps(report["validation"], indent=2) + "\n")
    for name, rows in report["generations"].items():
        f.write(f"\n{'='*88}\n{name}\n{'='*88}\n")
        for row in rows:
            f.write(f"[{row['category']}] Q: {row['prompt']}\nGREEDY: {row['greedy']}\nSAMPLED: {row['sampled']}\n\n")
print("Validation metrics:\n", json.dumps(report["validation"], indent=2))
print("\nReports:", txt_path, "and", json_path)
notify("TinyGPT SFT report ready", f"Saved {txt_path} and {json_path}")
del models; gc.collect(); torch.cuda.empty_cache()



FineWeb drift check: 977 chunks from /kaggle/input/notebooks/fakesiddharth/pretrain-stage-4/checkpoints/validation_cache.pt
  BASE_STAGE4 FineWeb validation loss: 3.6558
  MIX_A FineWeb validation loss: 4.2180
  MIX_B FineWeb validation loss: 3.9429
  Reference Stage 4 baseline: ~3.6472; measured base delta: +0.0086
Validation metrics:
 {
  "base_chat_loss": 3.288605055683657,
  "base_qa_loss": 3.4051075096250387,
  "mixA": {
    "val_chat_loss": 2.6180807887819117,
    "val_qa_loss": 3.3461873562482123
  },
  "mixB": {
    "val_chat_loss": 2.8018122288904683,
    "val_qa_loss": 3.1493803823127204
  }
}

Reports: /kaggle/working/stage4_sft_run_20261023/sft_comparison_report.txt and /kaggle/working/stage4_sft_run_20261023/sft_comparison_report.json
[ntfy] TinyGPT SFT report ready HTTP 200


## Outputs and handoff

All generated files belong to this `stage4/sft` run and are written under:

`/kaggle/working/stage4_sft_run_20261023/`

Expected files after successful completion:

- `mixA/sft_mixA_best.pt`, `mixA/sft_mixA_final.pt`, `mixA/sft_mixA_latest.pt`
- `mixB/sft_mixB_best.pt`, `mixB/sft_mixB_final.pt`, `mixB/sft_mixB_latest.pt`
- `sft_comparison_report.json` and `sft_comparison_report.txt`

The `best` checkpoints are the recommended starting point for evaluation. `latest` files include full training state for diagnosis; `best` and `final` are model weights. The notebook currently does not implement optimizer-state resume after interruption.

When finished, use **Save Version** with outputs enabled. Then attach this `stage4/sft` notebook version as an input to the testing notebook. Do not use outputs from an older SFT notebook by mistake.
